In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt #绘图工具
import seaborn as sns
import warnings
import random
import time
import torch
from sklearn.utils import shuffle
warnings.filterwarnings("ignore")
#设置jupyter显示多行结果
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all' #默认为'last'
# #显示所有列
pd.set_option('display.max_columns', None) #原来中间会有部分列的显示被省略
# #显示所有行
# pd.set_option('display.max_rows', None)
#设置value的显示长度为100，默认为50
pd.set_option('max_colwidth',100)
plt.rcParams['font.sans-serif'] = ['SimHei']  #显示中文
plt.rcParams['axes.unicode_minus']=False #用来正常显示负号

In [2]:
rootpath = "./KuaiRec 2.0/"

# 1. Process User Feature

In [4]:
user_features = pd.read_csv(rootpath + "data/user_features.csv")
user_features

,user_id,user_active_degree,is_lowactive_period,is_live_streamer,is_video_author,follow_user_num,follow_user_num_range,fans_user_num,fans_user_num_range,friend_user_num,friend_user_num_range,register_days,register_days_range,onehot_feat0,onehot_feat1,onehot_feat2,onehot_feat3,onehot_feat4,onehot_feat5,onehot_feat6,onehot_feat7,onehot_feat8,onehot_feat9,onehot_feat10,onehot_feat11,onehot_feat12,onehot_feat13,onehot_feat14,onehot_feat15,onehot_feat16,onehot_feat17
0,0,high_active,0,0,0,5,"(0,10]",0,0,0,0,107,61-90,0,1,17,638,2.0,0,1,6,184,6,3,0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,full_active,0,0,0,386,"(250,500]",4,"[1,10)",2,"[1,5)",327,181-365,0,3,25,1021,0.0,0,1,6,186,6,2,0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,full_active,0,0,0,27,"(10,50]",0,0,0,0,116,91-180,0,6,8,402,0.0,0,0,2,51,2,3,0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,full_active,0,0,0,16,"(10,50]",0,0,0,0,105,61-90,0,1,8,281,0.0,0,0,34,251,3,2,0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,full_active,0,0,0,122,"(100,150]",4,"[1,10)",0,0,225,181-365,0,1,8,316,1.0,0,1,46,99,4,2,0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7171,7171,full_active,0,0,1,52,"(50,100]",1,"[1,10)",0,0,283,181-365,0,3,8,302,1.0,0,1,15,259,1,4,0,1.0,0.0,0.0,0.0,0.0,0.0
7172,7172,full_active,0,0,0,45,"(10,50]",2,"[1,10)",2,"[1,5)",109,91-180,0,3,25,1025,0.0,0,1,35,11,2,0,0,1.0,0.0,0.0,0.0,0.0,0.0
7173,7173,full_active,0,0,0,615,500+,3,"[1,10)",2,"[1,5)",167,91-180,0,6,8,240,1.0,0,0,2,51,2,2,0,1.0,0.0,0.0,0.0,0.0,0.0
7174,7174,full_active,0,0,0,959,500+,0,0,0,0,241,181-365,1,6,25,1029,0.0,0,1,20,107,3,2,0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
#数据总缺失情况查阅
(user_features.isna().sum()/user_features.shape[0]).sort_values(ascending=False)

onehot_feat4             0.028010
onehot_feat12            0.010730
onehot_feat14            0.010452
onehot_feat13            0.010452
onehot_feat17            0.010312
onehot_feat16            0.010312
onehot_feat15            0.010312
onehot_feat3             0.000000
onehot_feat11            0.000000
onehot_feat10            0.000000
onehot_feat9             0.000000
onehot_feat8             0.000000
onehot_feat7             0.000000
onehot_feat6             0.000000
onehot_feat5             0.000000
user_id                  0.000000
user_active_degree       0.000000
onehot_feat1             0.000000
onehot_feat0             0.000000
register_days_range      0.000000
register_days            0.000000
friend_user_num_range    0.000000
friend_user_num          0.000000
fans_user_num_range      0.000000
fans_user_num            0.000000
follow_user_num_range    0.000000
follow_user_num          0.000000
is_video_author          0.000000
is_live_streamer         0.000000
is_lowactive_p

In [6]:
# 都是onehot缺失值，用0填补
user_features = user_features.fillna(0)
# 填补完再次查看是否有缺失值
(user_features.isna().sum()/user_features.shape[0]).sort_values(ascending=False)

user_id                  0.0
onehot_feat3             0.0
onehot_feat16            0.0
onehot_feat15            0.0
onehot_feat14            0.0
onehot_feat13            0.0
onehot_feat12            0.0
onehot_feat11            0.0
onehot_feat10            0.0
onehot_feat9             0.0
onehot_feat8             0.0
onehot_feat7             0.0
onehot_feat6             0.0
onehot_feat5             0.0
onehot_feat4             0.0
onehot_feat2             0.0
user_active_degree       0.0
onehot_feat1             0.0
onehot_feat0             0.0
register_days_range      0.0
register_days            0.0
friend_user_num_range    0.0
friend_user_num          0.0
fans_user_num_range      0.0
fans_user_num            0.0
follow_user_num_range    0.0
follow_user_num          0.0
is_video_author          0.0
is_live_streamer         0.0
is_lowactive_period      0.0
onehot_feat17            0.0
dtype: float64

## User feature

### sparse_feature: 
["user_active_degree","follow_user_num_range","fans_user_num_range","friend_user_num_range","register_days_range"] 需要进行labelencoder到0-n-1  
["onehot_feat" + str(i) for i in range(1,12)]

上述进行onehot展开

["onehot_feat0"] + ["onehot_feat" + str(i) for i in range(12,18)] + ["is_lowactive_period","is_live_streamer","is_video_author"]二值向量直接拼上去即可
### dense_feature: 
["follow_user_num","fans_user_num","friend_user_num","register_days"] 需要进行minmax归一化

In [7]:
user_sparse_feature_1 = ["user_active_degree","follow_user_num_range","fans_user_num_range","friend_user_num_range","register_days_range"]
user_sparse_feature_2 = ["onehot_feat" + str(i) for i in range(1,12)]
user_sparse_feature_3 = ["onehot_feat0"] + ["onehot_feat" + str(i) for i in range(12,18)] + ["is_lowactive_period","is_live_streamer","is_video_author"]
user_dense_feature = ["follow_user_num","fans_user_num","friend_user_num","register_days"]

In [8]:
# Label Encoding for sparse features,and do simple Transformation for dense features
from sklearn import preprocessing
for feat in user_sparse_feature_1:
    lbe = preprocessing.LabelEncoder()
    user_features[feat] = lbe.fit_transform(user_features[feat])

mms = preprocessing.MinMaxScaler(feature_range=(0, 1))
user_features[user_dense_feature] = mms.fit_transform(user_features[user_dense_feature])

user_features

,user_id,user_active_degree,is_lowactive_period,is_live_streamer,is_video_author,follow_user_num,follow_user_num_range,fans_user_num,fans_user_num_range,friend_user_num,friend_user_num_range,register_days,register_days_range,onehot_feat0,onehot_feat1,onehot_feat2,onehot_feat3,onehot_feat4,onehot_feat5,onehot_feat6,onehot_feat7,onehot_feat8,onehot_feat9,onehot_feat10,onehot_feat11,onehot_feat12,onehot_feat13,onehot_feat14,onehot_feat15,onehot_feat16,onehot_feat17
0,0,2,0,0,0,0.002381,0,0.000000,0,0.000000,0,0.047661,4,0,1,17,638,2.0,0,1,6,184,6,3,0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,1,0,0,0,0.183810,4,0.000351,1,0.001404,2,0.145657,1,0,3,25,1021,0.0,0,1,6,186,6,2,0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,1,0,0,0,0.012857,1,0.000000,0,0.000000,0,0.051670,6,0,6,8,402,0.0,0,0,2,51,2,3,0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,1,0,0,0,0.007619,1,0.000000,0,0.000000,0,0.046771,4,0,1,8,281,0.0,0,0,34,251,3,2,0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,1,0,0,0,0.058095,2,0.000351,1,0.000000,0,0.100223,1,0,1,8,316,1.0,0,1,46,99,4,2,0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7171,7171,1,0,0,1,0.024762,5,0.000088,1,0.000000,0,0.126058,1,0,3,8,302,1.0,0,1,15,259,1,4,0,1.0,0.0,0.0,0.0,0.0,0.0
7172,7172,1,0,0,0,0.021429,1,0.000175,1,0.001404,2,0.048552,6,0,3,25,1025,0.0,0,1,35,11,2,0,0,1.0,0.0,0.0,0.0,0.0,0.0
7173,7173,1,0,0,0,0.292857,7,0.000263,1,0.001404,2,0.074388,6,0,6,8,240,1.0,0,0,2,51,2,2,0,1.0,0.0,0.0,0.0,0.0,0.0
7174,7174,1,0,0,0,0.456667,7,0.000000,0,0.000000,0,0.107350,1,1,6,25,1029,0.0,0,1,20,107,3,2,0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# onehot sparse feature
user_features = pd.get_dummies(user_features, columns=user_sparse_feature_1+user_sparse_feature_2)
user_features

user_id  is_lowactive_period  is_live_streamer  is_video_author  \
0           0                    0                 0                0   
1           1                    0                 0                0   
2           2                    0                 0                0   
3           3                    0                 0                0   
4           4                    0                 0                0   
...       ...                  ...               ...              ...   
7171     7171                    0                 0                1   
7172     7172                    0                 0                0   
7173     7173                    0                 0                0   
7174     7174                    0                 0                0   
7175     7175                    0                 0                1   

      follow_user_num  fans_user_num  friend_user_num  register_days  \
0            0.002381       0.000000         0.000000       0.047661   
1            0.183810       0.000351         0.001404       0.145657   
2            0.012857       0.000000         0.000000       0.051670   
3            0.007619       0.000000         0.000000       0.046771   
4            0.058095       0.000351         0.000000       0.100223   
...               ...            ...              ...            ...   
7171         0.024762       0.000088         0.000000       0.126058   
7172         0.021429       0.000175         0.001404       0.048552   
7173         0.292857       0.000263         0.001404       0.074388   
7174         0.456667       0.000000         0.000000       0.107350   
7175         0.046667       0.003070         0.023158       0.074388   

      onehot_feat0  onehot_feat12  onehot_feat13  onehot_feat14  \
0                0            0.0            0.0            0.0   
1                0            0.0            0.0            0.0   
2                0            0.0            0.0            0.0   
3                0            0.0            0.0            0.0   
4                0            0.0            0.0            0.0   
...            ...            ...            ...            ...   
7171             0            1.0            0.0            0.0   
7172             0            1.0            0.0            0.0   
7173             0            1.0            0.0            0.0   
7174             1            0.0            0.0            0.0   
7175             0            0.0            0.0            0.0   

      onehot_feat15  onehot_feat16  onehot_feat17  user_active_degree_0  \
0               0.0            0.0            0.0                     0   
1               0.0            0.0            0.0                     0   
2               0.0            0.0            0.0                     0   
3               0.0            0.0            0.0                     0   
4               0.0            0.0            0.0                     0   
...             ...            ...            ...                   ...   
7171            0.0            0.0            0.0                     0   
7172            0.0            0.0            0.0                     0   
7173            0.0            0.0            0.0                     0   
7174            0.0            0.0            0.0                     0   
7175            0.0            0.0            0.0                     0   

      user_active_degree_1  user_active_degree_2  user_active_degree_3  \
0                        0                     1                     0   
1                        1                     0                     0   
2                        1                     0                     0   
3                        1                     0                     0   
4                        1                     0                     0   
...                    ...                   ...                   ...   
7171                     1              

In [10]:
user_features_columns = user_features.columns.tolist()
user_features_columns

['user_id',
 'is_lowactive_period',
 'is_live_streamer',
 'is_video_author',
 'follow_user_num',
 'fans_user_num',
 'friend_user_num',
 'register_days',
 'onehot_feat0',
 'onehot_feat12',
 'onehot_feat13',
 'onehot_feat14',
 'onehot_feat15',
 'onehot_feat16',
 'onehot_feat17',
 'user_active_degree_0',
 'user_active_degree_1',
 'user_active_degree_2',
 'user_active_degree_3',
 'follow_user_num_range_0',
 'follow_user_num_range_1',
 'follow_user_num_range_2',
 'follow_user_num_range_3',
 'follow_user_num_range_4',
 'follow_user_num_range_5',
 'follow_user_num_range_6',
 'follow_user_num_range_7',
 'fans_user_num_range_0',
 'fans_user_num_range_1',
 'fans_user_num_range_2',
 'fans_user_num_range_3',
 'fans_user_num_range_4',
 'fans_user_num_range_5',
 'fans_user_num_range_6',
 'friend_user_num_range_0',
 'friend_user_num_range_1',
 'friend_user_num_range_2',
 'friend_user_num_range_3',
 'friend_user_num_range_4',
 'friend_user_num_range_5',
 'friend_user_num_range_6',
 'register_days_rang

### PCA降维 到25维

In [11]:
from sklearn.decomposition import PCA
user_features_for_pca = user_features[user_features_columns[1:]]
user_features_for_pca
pca = PCA(n_components=25)
pca.fit(user_features_for_pca)
user_features_pca = pca.transform(user_features_for_pca)
user_features_for_pca.shape
user_features_pca.shape

is_lowactive_period  is_live_streamer  is_video_author  follow_user_num  \
0                       0                 0                0         0.002381   
1                       0                 0                0         0.183810   
2                       0                 0                0         0.012857   
3                       0                 0                0         0.007619   
4                       0                 0                0         0.058095   
...                   ...               ...              ...              ...   
7171                    0                 0                1         0.024762   
7172                    0                 0                0         0.021429   
7173                    0                 0                0         0.292857   
7174                    0                 0                0         0.456667   
7175                    0                 0                1         0.046667   

      fans_user_num  friend_user_num  register_days  onehot_feat0  \
0          0.000000         0.000000       0.047661             0   
1          0.000351         0.001404       0.145657             0   
2          0.000000         0.000000       0.051670             0   
3          0.000000         0.000000       0.046771             0   
4          0.000351         0.000000       0.100223             0   
...             ...              ...            ...           ...   
7171       0.000088         0.000000       0.126058             0   
7172       0.000175         0.001404       0.048552             0   
7173       0.000263         0.001404       0.074388             0   
7174       0.000000         0.000000       0.107350             1   
7175       0.003070         0.023158       0.074388             0   

      onehot_feat12  onehot_feat13  onehot_feat14  onehot_feat15  \
0               0.0            0.0            0.0            0.0   
1               0.0            0.0            0.0            0.0   
2               0.0            0.0            0.0            0.0   
3               0.0            0.0            0.0            0.0   
4               0.0            0.0            0.0            0.0   
...             ...            ...            ...            ...   
7171            1.0            0.0            0.0            0.0   
7172            1.0            0.0            0.0            0.0   
7173            1.0            0.0            0.0            0.0   
7174            0.0            0.0            0.0            0.0   
7175            0.0            0.0            0.0            0.0   

      onehot_feat16  onehot_feat17  user_active_degree_0  \
0               0.0            0.0                     0   
1               0.0            0.0                     0   
2               0.0            0.0                     0   
3               0.0            0.0                     0   
4               0.0            0.0                     0   
...             ...            ...                   ...   
7171            0.0            0.0                     0   
7172            0.0            0.0                     0   
7173            0.0            0.0                     0   
7174            0.0            0.0                     0   
7175            0.0            0.0                     0   

      user_active_degree_1  user_active_degree_2  user_active_degree_3  \
0                        0                     1                     0   
1                        1                     0                     0   
2                        1                     0                     0   
3                        1                     0                     0   
4                        1                     0                     0   
...                    ...                   ...                   ...   
7171                     1                     0                     0   
7172                     1                     0                     0   


PCA(n_components=25)

(7176, 1588)

(7176, 25)

In [12]:
new_user_data = pd.concat([user_features["user_id"], pd.DataFrame(user_features_pca)], axis=1)
new_user_data.columns = ["user_id"] + ["u"+str(i) for i in range(25)]
new_user_data

,user_id,u0,u1,u2,u3,u4,u5,u6,u7,u8,u9,u10,u11,u12,u13,u14,u15,u16,u17,u18,u19,u20,u21,u22,u23,u24
0,0,-1.674993,0.152371,-0.554478,0.847241,-0.487899,0.379005,0.038020,0.683137,-0.043919,0.463972,0.332833,-0.018468,0.682566,1.298489,0.160487,0.025993,-0.487937,-0.513339,-0.060585,0.092976,0.162210,-0.263889,-0.347352,0.251206,0.208874
1,1,0.353917,-1.110304,1.185043,0.134926,-0.119457,0.115978,-0.387008,0.258563,-0.518948,-0.095652,0.766431,-0.539570,0.089456,-0.006165,0.100268,0.734863,0.624212,-0.383560,-0.332877,-0.399949,0.062644,-0.418552,0.275639,0.188519,0.255468
2,2,-0.333553,1.103801,0.204809,0.232690,0.007381,0.132940,0.401279,-1.488611,-0.436226,0.569142,-0.651119,0.032094,-0.106102,-0.029214,0.112358,0.290853,0.251065,-0.164117,-0.360368,-0.423455,0.713594,0.097480,-0.051655,0.547016,0.350105
3,3,-0.658980,1.172114,0.762873,-0.289276,0.313181,-0.195914,-0.190670,-1.212718,-0.201246,0.093425,-0.309374,0.495787,0.410965,0.412080,-0.560811,-0.352097,0.541714,0.023365,-0.017354,0.213095,-0.118452,-0.016572,0.376097,-0.159158,-0.124515
4,4,-0.263066,-0.781668,-0.032065,-1.030237,-0.041411,-0.860592,-0.677270,-0.097401,-0.318732,0.046551,-0.447186,0.200679,0.159075,-0.143633,0.107536,0.197960,-0.430306,0.372218,0.136370,0.498894,-0.193749,-0.210194,-0.270706,0.178125,-0.176187
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7171,7171,0.404764,-0.846309,-0.645774,-0.188909,-0.138761,-0.583587,0.241269,-0.113076,-0.145269,0.295969,-0.890268,-0.350043,0.555424,-0.607487,0.392625,0.117326,-0.257680,-0.230332,0.268963,-0.653390,-0.647309,-0.405664,-0.040272,-0.325261,0.631514
7172,7172,0.138108,-0.873551,0.685209,0.142144,-0.605445,1.214768,0.451691,-0.694564,-0.120286,-0.455424,0.075029,-0.243325,-0.008977,-0.933601,-0.189219,0.429813,0.601006,-0.507452,-0.075559,-0.744555,0.350266,-0.340947,-0.065165,0.382320,0.235505
7173,7173,1.334196,0.262405,-0.146614,-1.131377,0.207926,0.824497,-0.787195,-0.134796,-0.132404,0.463317,-0.477721,-0.016490,0.220773,-0.522641,-0.074776,0.033094,0.299075,-0.276668,0.104060,-0.676064,0.904495,0.089272,-0.141311,0.509480,0.112142
7174,7174,-0.423911,-0.368874,0.673537,0.627725,1.059992,-0.631193,-0.119376,0.054937,0.098089,-0.610454,-0.362799,-0.476146,-0.129339,0.257187,-0.328869,0.313102,0.809837,0.353500,0.113543,-0.264847,0.908311,0.635354,0.354454,-0.537803,-0.327043


In [13]:
new_user_data.to_csv("./user_feature_pca.csv", sep=" ", index=False)

# 2. Process Item Feature

In [7]:
item_daily_features = pd.read_csv(rootpath + "data/item_daily_features.csv")
item_daily_features

,video_id,date,author_id,video_type,upload_dt,upload_type,visible_status,video_duration,video_width,video_height,music_id,video_tag_id,video_tag_name,show_cnt,show_user_num,play_cnt,play_user_num,play_duration,complete_play_cnt,complete_play_user_num,valid_play_cnt,valid_play_user_num,long_time_play_cnt,long_time_play_user_num,short_time_play_cnt,short_time_play_user_num,play_progress,comment_stay_duration,like_cnt,like_user_num,click_like_cnt,double_click_cnt,cancel_like_cnt,cancel_like_user_num,comment_cnt,comment_user_num,direct_comment_cnt,reply_comment_cnt,delete_comment_cnt,delete_comment_user_num,comment_like_cnt,comment_like_user_num,follow_cnt,follow_user_num,cancel_follow_cnt,cancel_follow_user_num,share_cnt,share_user_num,download_cnt,download_user_num,report_cnt,report_user_num,reduce_similar_cnt,reduce_similar_user_num,collect_cnt,collect_user_num,cancel_collect_cnt,cancel_collect_user_num
0,0,20200705,3309,NORMAL,2020-03-30,ShortImport,public,5966.0,720,1280,3350323409,841,建筑,14665,11372,10141,7485,88729488,5657,4834,5503,4775,5503,4775,1939,1481,0.799860,6629173,573,569,315,257,87,85,11,11,8,3,0,0,112,61,284,284,0,0,2,2,8,8,0,0,3,3,NaN,NaN,NaN,NaN
1,0,20200706,3309,NORMAL,2020-03-30,ShortImport,public,5966.0,720,1280,3350323409,841,建筑,10883,8513,7321,5490,64264607,4162,3522,4039,3468,4039,3468,1340,1040,0.805253,3997498,302,301,159,142,47,47,7,7,6,1,0,0,60,32,201,200,0,0,1,1,2,2,0,0,5,5,NaN,NaN,NaN,NaN
2,0,20200707,3309,NORMAL,2020-03-30,ShortImport,public,5966.0,720,1280,3350323409,841,建筑,7842,6281,4757,3724,41338741,2734,2403,2640,2376,2640,2376,866,683,0.808821,3314323,205,205,121,84,52,50,4,3,3,1,0,0,59,26,131,131,0,0,1,1,2,2,0,0,0,0,NaN,NaN,NaN,NaN
3,0,20200708,3309,NORMAL,2020-03-30,ShortImport,public,5966.0,720,1280,3350323409,841,建筑,8916,7229,5172,3961,45281254,2950,2525,2865,2498,2865,2498,977,765,0.801680,4235579,297,293,178,119,60,59,4,4,2,2,0,0,91,46,179,179,0,0,2,2,3,3,0,0,3,3,NaN,NaN,NaN,NaN
4,0,20200709,3309,NORMAL,2020-03-30,ShortImport,public,5966.0,720,1280,3350323409,841,建筑,8502,6658,5392,3946,46952744,3058,2566,2946,2533,2945,2532,1046,752,0.805359,3862095,307,305,166,141,57,56,5,2,0,5,0,0,76,47,186,186,0,0,0,0,2,2,2,1,1,1,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
343336,10723,20200905,236,NORMAL,2020-09-05,ShortImport,public,4833.0,720,1280,4428603493,11,生活,277,173,214,157,1681908,117,106,114,104,114,104,83,66,0.596591,337534,24,24,6,18,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0
343337,10724,20200905,5271,NORMAL,2020-09-05,LongImport,public,54720.0,720,1280,1090207430,2,音乐,1100,1017,965,856,56090732,535,523,754,721,657,637,113,99,0.574591,1249884,264,261,224,39,14,13,29,27,29,0,0,0,3,3,8,8,0,0,1,1,1,1,0,0,0,0,0.0,0.0,0.0,0.0
343338,10725,20200905,1924,NORMAL,2020-09-05,ShortImport,public,15800.0,576,1024,4429406509,15,才艺,16996,16345,15487,14672,323787284,8149,8015,9317,9171,7949,7871,4342,4139,0.577613,1963153,851,845,574,275,19,19,36,28,34,2,0,0,1,1,12,12,0,0,3,2,5,5,0,0,4,4,0.0,0.0,0.0,0.0
343339,10726,20200905,7604,NORMAL,2020-09-05,ShortImport,public,5132.0,528,960,68154,19,情感,7644,7568,7859,7480,128835301,5480,5395,5382,5319,5382,5319,1648,1580,0.818123,192695,44,44,18,25,1,1,0,0,0,0,0,0,0,0,2,2,0,0,1,1,2,2,0,0,1,1,0.0,0.0,0.0,0.0


In [8]:
item_daily_features.video_tag_name

0         建筑
1         建筑
2         建筑
3         建筑
4         建筑
          ..
343336    生活
343337    音乐
343338    才艺
343339    情感
343340    穿搭
Name: video_tag_name, Length: 343341, dtype: object

In [9]:
#数据总缺失情况查阅
(item_daily_features.isna().sum()/item_daily_features.shape[0]).sort_values(ascending=False)

cancel_collect_user_num     0.202956
cancel_collect_cnt          0.202956
collect_user_num            0.202956
collect_cnt                 0.202956
video_tag_name              0.094466
video_duration              0.030867
reply_comment_cnt           0.000000
comment_like_cnt            0.000000
delete_comment_user_num     0.000000
delete_comment_cnt          0.000000
comment_cnt                 0.000000
direct_comment_cnt          0.000000
comment_user_num            0.000000
follow_cnt                  0.000000
cancel_like_user_num        0.000000
cancel_like_cnt             0.000000
comment_like_user_num       0.000000
cancel_follow_user_num      0.000000
follow_user_num             0.000000
cancel_follow_cnt           0.000000
click_like_cnt              0.000000
share_cnt                   0.000000
share_user_num              0.000000
download_cnt                0.000000
download_user_num           0.000000
report_cnt                  0.000000
report_user_num             0.000000
r

In [10]:
item_daily_features.columns

Index(['video_id', 'date', 'author_id', 'video_type', 'upload_dt',
       'upload_type', 'visible_status', 'video_duration', 'video_width',
       'video_height', 'music_id', 'video_tag_id', 'video_tag_name',
       'show_cnt', 'show_user_num', 'play_cnt', 'play_user_num',
       'play_duration', 'complete_play_cnt', 'complete_play_user_num',
       'valid_play_cnt', 'valid_play_user_num', 'long_time_play_cnt',
       'long_time_play_user_num', 'short_time_play_cnt',
       'short_time_play_user_num', 'play_progress', 'comment_stay_duration',
       'like_cnt', 'like_user_num', 'click_like_cnt', 'double_click_cnt',
       'cancel_like_cnt', 'cancel_like_user_num', 'comment_cnt',
       'comment_user_num', 'direct_comment_cnt', 'reply_comment_cnt',
       'delete_comment_cnt', 'delete_comment_user_num', 'comment_like_cnt',
       'comment_like_user_num', 'follow_cnt', 'follow_user_num',
       'cancel_follow_cnt', 'cancel_follow_user_num', 'share_cnt',
       'share_user_num', 'download

## Item Daily feature

对缺失较多的['collect_cnt', 'collect_user_num', 'cancel_collect_cnt','cancel_collect_user_num']特征直接丢弃  
video_duration缺失值用均值填充  
video_tag_id和video_tag_name重复，用video_tag_id即可，video_tag_name可以用来可视化  

### sparse_feature: 
['author_id','video_tag_id','upload_type','visible_status','music_id'] 需要进行labelencoder到0-n-1  
['author_id','music_id']重复度很小，可以考虑不用加进去

上述进行onehot展开

["video_type"] 转为二值向量直接拼上去即可
### dense_feature: 
['video_duration','video_width','video_height','show_cnt', 'show_user_num', 'play_cnt', 'play_user_num','play_duration', 'complete_play_cnt', 'complete_play_user_num','valid_play_cnt', 'valid_play_user_num', 'long_time_play_cnt','long_time_play_user_num', 'short_time_play_cnt','short_time_play_user_num', 'play_progress', 'comment_stay_duration','like_cnt', 'like_user_num', 'click_like_cnt', 'double_click_cnt','cancel_like_cnt', 'cancel_like_user_num', 'comment_cnt','comment_user_num', 'direct_comment_cnt', 'reply_comment_cnt','delete_comment_cnt', 'delete_comment_user_num', 'comment_like_cnt','comment_like_user_num', 'follow_cnt', 'follow_user_num','cancel_follow_cnt', 'cancel_follow_user_num', 'share_cnt','share_user_num', 'download_cnt', 'download_user_num', 'report_cnt','report_user_num', 'reduce_similar_cnt', 'reduce_similar_user_num'] 需要进行minmax归一化

In [11]:
item_daily_features['video_duration'] = item_daily_features['video_duration'].fillna(item_daily_features['video_duration'].mean())

In [12]:
item_sparse_feature_1 = ['video_tag_id','upload_type','visible_status'] 
item_sparse_feature_2 = ["video_type"]
item_dense_feature = ['video_duration','video_width','video_height','show_cnt', 'show_user_num', 'play_cnt', 'play_user_num','play_duration', 'complete_play_cnt', 'complete_play_user_num','valid_play_cnt', 'valid_play_user_num', 'long_time_play_cnt','long_time_play_user_num', 'short_time_play_cnt','short_time_play_user_num', 'play_progress', 'comment_stay_duration','like_cnt', 'like_user_num', 'click_like_cnt', 'double_click_cnt','cancel_like_cnt', 'cancel_like_user_num', 'comment_cnt','comment_user_num', 'direct_comment_cnt', 'reply_comment_cnt','delete_comment_cnt', 'delete_comment_user_num', 'comment_like_cnt','comment_like_user_num', 'follow_cnt', 'follow_user_num','cancel_follow_cnt', 'cancel_follow_user_num', 'share_cnt','share_user_num', 'download_cnt', 'download_user_num', 'report_cnt','report_user_num', 'reduce_similar_cnt', 'reduce_similar_user_num'] 

In [13]:
# Label Encoding for sparse features,and do simple Transformation for dense features
from sklearn import preprocessing
for feat in item_sparse_feature_1 + item_sparse_feature_2:
    lbe = preprocessing.LabelEncoder()
    item_daily_features[feat] = lbe.fit_transform(item_daily_features[feat])

mms = preprocessing.MinMaxScaler(feature_range=(0, 1))
item_daily_features[item_dense_feature] = mms.fit_transform(item_daily_features[item_dense_feature])

item_daily_features

,video_id,date,author_id,video_type,upload_dt,upload_type,visible_status,video_duration,video_width,video_height,music_id,video_tag_id,video_tag_name,show_cnt,show_user_num,play_cnt,play_user_num,play_duration,complete_play_cnt,complete_play_user_num,valid_play_cnt,valid_play_user_num,long_time_play_cnt,long_time_play_user_num,short_time_play_cnt,short_time_play_user_num,play_progress,comment_stay_duration,like_cnt,like_user_num,click_like_cnt,double_click_cnt,cancel_like_cnt,cancel_like_user_num,comment_cnt,comment_user_num,direct_comment_cnt,reply_comment_cnt,delete_comment_cnt,delete_comment_user_num,comment_like_cnt,comment_like_user_num,follow_cnt,follow_user_num,cancel_follow_cnt,cancel_follow_user_num,share_cnt,share_user_num,download_cnt,download_user_num,report_cnt,report_user_num,reduce_similar_cnt,reduce_similar_user_num,collect_cnt,collect_user_num,cancel_collect_cnt,cancel_collect_user_num
0,0,20200705,3309,1,2020-03-30,15,2,0.018813,0.163399,0.355030,3350323409,169,建筑,0.000348,0.000292,0.000246,0.000201,4.467997e-05,0.000224,0.000199,0.000189,0.000169,0.000208,0.000185,0.000186,0.000162,0.799860,3.179897e-05,0.000214,0.000215,1.850155e-04,0.000266,0.000372,0.000407,0.000060,0.000064,0.000046,0.00003,0.0,0.0,0.000164,0.000199,0.000575,0.000576,0.0,0.0,0.000007,0.000007,0.000060,0.000060,0.000000,0.000000,0.000087,0.000089,NaN,NaN,NaN,NaN
1,0,20200706,3309,1,2020-03-30,15,2,0.018813,0.163399,0.355030,3350323409,169,建筑,0.000258,0.000219,0.000178,0.000147,3.236061e-05,0.000165,0.000145,0.000138,0.000123,0.000152,0.000134,0.000128,0.000114,0.805253,1.917529e-05,0.000113,0.000114,9.338878e-05,0.000147,0.000201,0.000225,0.000038,0.000041,0.000034,0.00001,0.0,0.0,0.000088,0.000104,0.000407,0.000406,0.0,0.0,0.000003,0.000004,0.000015,0.000015,0.000000,0.000000,0.000145,0.000148,NaN,NaN,NaN,NaN
2,0,20200707,3309,1,2020-03-30,15,2,0.018813,0.163399,0.355030,3350323409,169,建筑,0.000186,0.000161,0.000116,0.000100,2.081623e-05,0.000108,0.000099,0.000090,0.000084,0.000100,0.000092,0.000083,0.000075,0.808821,1.589822e-05,0.000077,0.000077,7.106945e-05,0.000087,0.000223,0.000239,0.000022,0.000017,0.000017,0.00001,0.0,0.0,0.000086,0.000085,0.000265,0.000266,0.0,0.0,0.000003,0.000004,0.000015,0.000015,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
3,0,20200708,3309,1,2020-03-30,15,2,0.018813,0.163399,0.355030,3350323409,169,建筑,0.000211,0.000186,0.000126,0.000106,2.280150e-05,0.000117,0.000104,0.000098,0.000088,0.000108,0.000097,0.000094,0.000084,0.801680,2.031732e-05,0.000111,0.000110,1.045484e-04,0.000123,0.000257,0.000282,0.000022,0.000023,0.000011,0.00002,0.0,0.0,0.000133,0.000150,0.000363,0.000363,0.0,0.0,0.000007,0.000007,0.000022,0.000022,0.000000,0.000000,0.000087,0.000089,NaN,NaN,NaN,NaN
4,0,20200709,3309,1,2020-03-30,15,2,0.018813,0.163399,0.355030,3350323409,169,建筑,0.000202,0.000171,0.000131,0.000106,2.364318e-05,0.000121,0.000106,0.000101,0.000090,0.000111,0.000098,0.000100,0.000082,0.805359,1.852579e-05,0.000115,0.000115,9.750023e-05,0.000146,0.000244,0.000268,0.000027,0.000012,0.000000,0.00005,0.0,0.0,0.000111,0.000153,0.000377,0.000377,0.0,0.0,0.000000,0.000000,0.000015,0.000015,0.010101,0.005682,0.000029,0.000030,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
343336,10723,20200905,236,1,2020-09-05,15,2,0.015216,0.163399,0.355030,4428603493,12,生活,0.000007,0.000004,0.000005,0.000004,8.469292e-07,0.000005,0.000004,0.000004,0.000004,0.000004,0.000004,0.000008,0.000007,0.596591,1.619091e-06,0.000009,0.000009,3.524105e-06,0.000019,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.0,0.0,0.000001,0.000003,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
343337,10724,20200905,5271,1,2020-09-05,7,2,0.173587,0.163399,0.355030,1090207430,3,音乐,0.000026,0.0

In [14]:
# onehot sparse feature
item_daily_features_onehot = pd.get_dummies(item_daily_features[item_sparse_feature_1], columns=item_sparse_feature_1)
item_daily_features_onehot

,video_tag_id_0,video_tag_id_1,video_tag_id_2,video_tag_id_3,video_tag_id_4,video_tag_id_5,video_tag_id_6,video_tag_id_7,video_tag_id_8,video_tag_id_9,video_tag_id_10,video_tag_id_11,video_tag_id_12,video_tag_id_13,video_tag_id_14,video_tag_id_15,video_tag_id_16,video_tag_id_17,video_tag_id_18,video_tag_id_19,video_tag_id_20,video_tag_id_21,video_tag_id_22,video_tag_id_23,video_tag_id_24,video_tag_id_25,video_tag_id_26,video_tag_id_27,video_tag_id_28,video_tag_id_29,video_tag_id_30,video_tag_id_31,video_tag_id_32,video_tag_id_33,video_tag_id_34,video_tag_id_35,video_tag_id_36,video_tag_id_37,video_tag_id_38,video_tag_id_39,video_tag_id_40,video_tag_id_41,video_tag_id_42,video_tag_id_43,video_tag_id_44,video_tag_id_45,video_tag_id_46,video_tag_id_47,video_tag_id_48,video_tag_id_49,video_tag_id_50,video_tag_id_51,video_tag_id_52,video_tag_id_53,video_tag_id_54,video_tag_id_55,video_tag_id_56,video_tag_id_57,video_tag_id_58,video_tag_id_59,video_tag_id_60,video_tag_id_61,video_tag_id_62,video_tag_id_63,video_tag_id_64,video_tag_id_65,video_tag_id_66,video_tag_id_67,video_tag_id_68,video_tag_id_69,video_tag_id_70,video_tag_id_71,video_tag_id_72,video_tag_id_73,video_tag_id_74,video_tag_id_75,video_tag_id_76,video_tag_id_77,video_tag_id_78,video_tag_id_79,video_tag_id_80,video_tag_id_81,video_tag_id_82,video_tag_id_83,video_tag_id_84,video_tag_id_85,video_tag_id_86,video_tag_id_87,video_tag_id_88,video_tag_id_89,video_tag_id_90,video_tag_id_91,video_tag_id_92,video_tag_id_93,video_tag_id_94,video_tag_id_95,video_tag_id_96,video_tag_id_97,video_tag_id_98,video_tag_id_99,video_tag_id_100,video_tag_id_101,video_tag_id_102,video_tag_id_103,video_tag_id_104,video_tag_id_105,video_tag_id_106,video_tag_id_107,video_tag_id_108,video_tag_id_109,video_tag_id_110,video_tag_id_111,video_tag_id_112,video_tag_id_113,video_tag_id_114,video_tag_id_115,video_tag_id_116,video_tag_id_117,video_tag_id_118,video_tag_id_119,video_tag_id_120,video_tag_id_121,video_tag_id_122,video_tag_id_123,video_tag_id_124,video_tag_id_125,video_tag_id_126,video_tag_id_127,video_tag_id_128,video_tag_id_129,video_tag_id_130,video_tag_id_131,video_tag_id_132,video_tag_id_133,video_tag_id_134,video_tag_id_135,video_tag_id_136,video_tag_id_137,video_tag_id_138,video_tag_id_139,video_tag_id_140,video_tag_id_141,video_tag_id_142,video_tag_id_143,video_tag_id_144,video_tag_id_145,video_tag_id_146,video_tag_id_147,video_tag_id_148,video_tag_id_149,video_tag_id_150,video_tag_id_151,video_tag_id_152,video_tag_id_153,video_tag_id_154,video_tag_id_155,video_tag_id_156,video_tag_id_157,video_tag_id_158,video_tag_id_159,video_tag_id_160,video_tag_id_161,video_tag_id_162,video_tag_id_163,video_tag_id_164,video_tag_id_165,video_tag_id_166,video_tag_id_167,video_tag_id_168,video_tag_id_169,video_tag_id_170,video_tag_id_171,video_tag_id_172,video_tag_id_173,video_tag_id_174,video_tag_id_175,video_tag_id_176,video_tag_id_177,video_tag_id_178,video_tag_id_179,video_tag_id_180,video_tag_id_181,video_tag_id_182,video_tag_id_183,video_tag_id_184,video_tag_id_185,video_tag_id_186,video_tag_id_187,video_tag_id_188,video_tag_id_189,video_tag_id_190,video_tag_id_191,video_tag_id_192,video_tag_id_193,video_tag_id_194,video_tag_id_195,video_tag_id_196,video_tag_id_197,video_tag_id_198,video_tag_id_199,video_tag_id_200,video_tag_id_201,video_tag_id_202,video_tag_id_203,video_tag_id_204,video_tag_id_205,video_tag_id_206,video_tag_id_207,video_tag_id_208,video_tag_id_209,video_tag_id_210,video_tag_id_211,video_tag_id_212,video_tag_id_213,video_tag_id_214,video_tag_id_215,video_tag_id_216,video_tag_id_217,video_tag_id_218,video_tag_id_219,video_tag_id_220,video_tag_id_221,video_tag_id_222,video_tag_id_223,video_tag_id_224,video_tag_id_225,video_tag_id_226,video_tag_id_227,video_tag_id_228,video_tag_id_229,video_tag_id_230,video_tag_id_231,video_tag_id_232,video_tag_id_233,video_tag_id_234,video_tag_id_235,video_tag_id_236,video_tag_id_237,video_tag_id_238,video_tag_id_239,video_tag_id_240,video_tag_id

In [15]:
item_daily_features_processd = pd.concat([item_daily_features[['video_id','date']+item_sparse_feature_2+item_dense_feature],item_daily_features_onehot],axis=1)
item_daily_features_processd

,video_id,date,video_type,video_duration,video_width,video_height,show_cnt,show_user_num,play_cnt,play_user_num,play_duration,complete_play_cnt,complete_play_user_num,valid_play_cnt,valid_play_user_num,long_time_play_cnt,long_time_play_user_num,short_time_play_cnt,short_time_play_user_num,play_progress,comment_stay_duration,like_cnt,like_user_num,click_like_cnt,double_click_cnt,cancel_like_cnt,cancel_like_user_num,comment_cnt,comment_user_num,direct_comment_cnt,reply_comment_cnt,delete_comment_cnt,delete_comment_user_num,comment_like_cnt,comment_like_user_num,follow_cnt,follow_user_num,cancel_follow_cnt,cancel_follow_user_num,share_cnt,share_user_num,download_cnt,download_user_num,report_cnt,report_user_num,reduce_similar_cnt,reduce_similar_user_num,video_tag_id_0,video_tag_id_1,video_tag_id_2,video_tag_id_3,video_tag_id_4,video_tag_id_5,video_tag_id_6,video_tag_id_7,video_tag_id_8,video_tag_id_9,video_tag_id_10,video_tag_id_11,video_tag_id_12,video_tag_id_13,video_tag_id_14,video_tag_id_15,video_tag_id_16,video_tag_id_17,video_tag_id_18,video_tag_id_19,video_tag_id_20,video_tag_id_21,video_tag_id_22,video_tag_id_23,video_tag_id_24,video_tag_id_25,video_tag_id_26,video_tag_id_27,video_tag_id_28,video_tag_id_29,video_tag_id_30,video_tag_id_31,video_tag_id_32,video_tag_id_33,video_tag_id_34,video_tag_id_35,video_tag_id_36,video_tag_id_37,video_tag_id_38,video_tag_id_39,video_tag_id_40,video_tag_id_41,video_tag_id_42,video_tag_id_43,video_tag_id_44,video_tag_id_45,video_tag_id_46,video_tag_id_47,video_tag_id_48,video_tag_id_49,video_tag_id_50,video_tag_id_51,video_tag_id_52,video_tag_id_53,video_tag_id_54,video_tag_id_55,video_tag_id_56,video_tag_id_57,video_tag_id_58,video_tag_id_59,video_tag_id_60,video_tag_id_61,video_tag_id_62,video_tag_id_63,video_tag_id_64,video_tag_id_65,video_tag_id_66,video_tag_id_67,video_tag_id_68,video_tag_id_69,video_tag_id_70,video_tag_id_71,video_tag_id_72,video_tag_id_73,video_tag_id_74,video_tag_id_75,video_tag_id_76,video_tag_id_77,video_tag_id_78,video_tag_id_79,video_tag_id_80,video_tag_id_81,video_tag_id_82,video_tag_id_83,video_tag_id_84,video_tag_id_85,video_tag_id_86,video_tag_id_87,video_tag_id_88,video_tag_id_89,video_tag_id_90,video_tag_id_91,video_tag_id_92,video_tag_id_93,video_tag_id_94,video_tag_id_95,video_tag_id_96,video_tag_id_97,video_tag_id_98,video_tag_id_99,video_tag_id_100,video_tag_id_101,video_tag_id_102,video_tag_id_103,video_tag_id_104,video_tag_id_105,video_tag_id_106,video_tag_id_107,video_tag_id_108,video_tag_id_109,video_tag_id_110,video_tag_id_111,video_tag_id_112,video_tag_id_113,video_tag_id_114,video_tag_id_115,video_tag_id_116,video_tag_id_117,video_tag_id_118,video_tag_id_119,video_tag_id_120,video_tag_id_121,video_tag_id_122,video_tag_id_123,video_tag_id_124,video_tag_id_125,video_tag_id_126,video_tag_id_127,video_tag_id_128,video_tag_id_129,video_tag_id_130,video_tag_id_131,video_tag_id_132,video_tag_id_133,video_tag_id_134,video_tag_id_135,video_tag_id_136,video_tag_id_137,video_tag_id_138,video_tag_id_139,video_tag_id_140,video_tag_id_141,video_tag_id_142,video_tag_id_143,video_tag_id_144,video_tag_id_145,video_tag_id_146,video_tag_id_147,video_tag_id_148,video_tag_id_149,video_tag_id_150,video_tag_id_151,video_tag_id_152,video_tag_id_153,video_tag_id_154,video_tag_id_155,video_tag_id_156,video_tag_id_157,video_tag_id_158,video_tag_id_159,video_tag_id_160,video_tag_id_161,video_tag_id_162,video_tag_id_163,video_tag_id_164,video_tag_id_165,video_tag_id_166,video_tag_id_167,video_tag_id_168,video_tag_id_169,video_tag_id_170,video_tag_id_171,video_tag_id_172,video_tag_id_173,video_tag_id_174,video_tag_id_175,video_tag_id_176,video_tag_id_177,video_tag_id_178,video_tag_id_179,video_tag_id_180,video_tag_id_181,video_tag_id_182,video_tag_id_183,video_tag_id_184,video_tag_id_185,video_tag_id_186,video_tag_id_187,video_tag_id_188,video_tag_id_189,video_tag_id_190,video_tag_id_191,video_tag_id_192,video_tag_id_193,video_tag_id_194,video_tag_id_195,video_tag_id_196,v

In [22]:
# item daily feature 特征是统计了每个item每天的特征，这个其实就对应了分布的漂移

In [16]:
item_features_columns = item_daily_features_processd.columns.tolist()
item_features_columns

['video_id',
 'date',
 'video_type',
 'video_duration',
 'video_width',
 'video_height',
 'show_cnt',
 'show_user_num',
 'play_cnt',
 'play_user_num',
 'play_duration',
 'complete_play_cnt',
 'complete_play_user_num',
 'valid_play_cnt',
 'valid_play_user_num',
 'long_time_play_cnt',
 'long_time_play_user_num',
 'short_time_play_cnt',
 'short_time_play_user_num',
 'play_progress',
 'comment_stay_duration',
 'like_cnt',
 'like_user_num',
 'click_like_cnt',
 'double_click_cnt',
 'cancel_like_cnt',
 'cancel_like_user_num',
 'comment_cnt',
 'comment_user_num',
 'direct_comment_cnt',
 'reply_comment_cnt',
 'delete_comment_cnt',
 'delete_comment_user_num',
 'comment_like_cnt',
 'comment_like_user_num',
 'follow_cnt',
 'follow_user_num',
 'cancel_follow_cnt',
 'cancel_follow_user_num',
 'share_cnt',
 'share_user_num',
 'download_cnt',
 'download_user_num',
 'report_cnt',
 'report_user_num',
 'reduce_similar_cnt',
 'reduce_similar_user_num',
 'video_tag_id_0',
 'video_tag_id_1',
 'video_tag_id_

### PCA降维 到25维

In [17]:
from sklearn.decomposition import PCA
item_daily_features_processd_for_pca = item_daily_features_processd[item_features_columns[2:]]
item_daily_features_processd_for_pca
pca = PCA(n_components=25)
pca.fit(item_daily_features_processd_for_pca)
item_daily_features_processd_pca = pca.transform(item_daily_features_processd_for_pca)
item_daily_features_processd_for_pca.shape
item_daily_features_processd_pca.shape

,video_type,video_duration,video_width,video_height,show_cnt,show_user_num,play_cnt,play_user_num,play_duration,complete_play_cnt,complete_play_user_num,valid_play_cnt,valid_play_user_num,long_time_play_cnt,long_time_play_user_num,short_time_play_cnt,short_time_play_user_num,play_progress,comment_stay_duration,like_cnt,like_user_num,click_like_cnt,double_click_cnt,cancel_like_cnt,cancel_like_user_num,comment_cnt,comment_user_num,direct_comment_cnt,reply_comment_cnt,delete_comment_cnt,delete_comment_user_num,comment_like_cnt,comment_like_user_num,follow_cnt,follow_user_num,cancel_follow_cnt,cancel_follow_user_num,share_cnt,share_user_num,download_cnt,download_user_num,report_cnt,report_user_num,reduce_similar_cnt,reduce_similar_user_num,video_tag_id_0,video_tag_id_1,video_tag_id_2,video_tag_id_3,video_tag_id_4,video_tag_id_5,video_tag_id_6,video_tag_id_7,video_tag_id_8,video_tag_id_9,video_tag_id_10,video_tag_id_11,video_tag_id_12,video_tag_id_13,video_tag_id_14,video_tag_id_15,video_tag_id_16,video_tag_id_17,video_tag_id_18,video_tag_id_19,video_tag_id_20,video_tag_id_21,video_tag_id_22,video_tag_id_23,video_tag_id_24,video_tag_id_25,video_tag_id_26,video_tag_id_27,video_tag_id_28,video_tag_id_29,video_tag_id_30,video_tag_id_31,video_tag_id_32,video_tag_id_33,video_tag_id_34,video_tag_id_35,video_tag_id_36,video_tag_id_37,video_tag_id_38,video_tag_id_39,video_tag_id_40,video_tag_id_41,video_tag_id_42,video_tag_id_43,video_tag_id_44,video_tag_id_45,video_tag_id_46,video_tag_id_47,video_tag_id_48,video_tag_id_49,video_tag_id_50,video_tag_id_51,video_tag_id_52,video_tag_id_53,video_tag_id_54,video_tag_id_55,video_tag_id_56,video_tag_id_57,video_tag_id_58,video_tag_id_59,video_tag_id_60,video_tag_id_61,video_tag_id_62,video_tag_id_63,video_tag_id_64,video_tag_id_65,video_tag_id_66,video_tag_id_67,video_tag_id_68,video_tag_id_69,video_tag_id_70,video_tag_id_71,video_tag_id_72,video_tag_id_73,video_tag_id_74,video_tag_id_75,video_tag_id_76,video_tag_id_77,video_tag_id_78,video_tag_id_79,video_tag_id_80,video_tag_id_81,video_tag_id_82,video_tag_id_83,video_tag_id_84,video_tag_id_85,video_tag_id_86,video_tag_id_87,video_tag_id_88,video_tag_id_89,video_tag_id_90,video_tag_id_91,video_tag_id_92,video_tag_id_93,video_tag_id_94,video_tag_id_95,video_tag_id_96,video_tag_id_97,video_tag_id_98,video_tag_id_99,video_tag_id_100,video_tag_id_101,video_tag_id_102,video_tag_id_103,video_tag_id_104,video_tag_id_105,video_tag_id_106,video_tag_id_107,video_tag_id_108,video_tag_id_109,video_tag_id_110,video_tag_id_111,video_tag_id_112,video_tag_id_113,video_tag_id_114,video_tag_id_115,video_tag_id_116,video_tag_id_117,video_tag_id_118,video_tag_id_119,video_tag_id_120,video_tag_id_121,video_tag_id_122,video_tag_id_123,video_tag_id_124,video_tag_id_125,video_tag_id_126,video_tag_id_127,video_tag_id_128,video_tag_id_129,video_tag_id_130,video_tag_id_131,video_tag_id_132,video_tag_id_133,video_tag_id_134,video_tag_id_135,video_tag_id_136,video_tag_id_137,video_tag_id_138,video_tag_id_139,video_tag_id_140,video_tag_id_141,video_tag_id_142,video_tag_id_143,video_tag_id_144,video_tag_id_145,video_tag_id_146,video_tag_id_147,video_tag_id_148,video_tag_id_149,video_tag_id_150,video_tag_id_151,video_tag_id_152,video_tag_id_153,video_tag_id_154,video_tag_id_155,video_tag_id_156,video_tag_id_157,video_tag_id_158,video_tag_id_159,video_tag_id_160,video_tag_id_161,video_tag_id_162,video_tag_id_163,video_tag_id_164,video_tag_id_165,video_tag_id_166,video_tag_id_167,video_tag_id_168,video_tag_id_169,video_tag_id_170,video_tag_id_171,video_tag_id_172,video_tag_id_173,video_tag_id_174,video_tag_id_175,video_tag_id_176,video_tag_id_177,video_tag_id_178,video_tag_id_179,video_tag_id_180,video_tag_id_181,video_tag_id_182,video_tag_id_183,video_tag_id_184,video_tag_id_185,video_tag_id_186,video_tag_id_187,video_tag_id_188,video_tag_id_189,video_tag_id_190,video_tag_id_191,video_tag_id_192,video_tag_id_193,video_tag_id_194,video_tag_id_195,video_tag_id_196,video_tag_id_19

PCA(n_components=25)

(343341, 628)

(343341, 25)

In [25]:
new_item_daily_data = pd.concat([item_daily_features_processd[['video_id', 'date']], pd.DataFrame(item_daily_features_processd_pca)], axis=1)
new_item_daily_data.columns = ['video_id', 'date'] + ["v"+str(i) for i in range(25)]
new_item_daily_data

,video_id,date,v0,v1,v2,v3,v4,v5,v6,v7,v8,v9,v10,v11,v12,v13,v14,v15,v16,v17,v18,v19,v20,v21,v22,v23,v24
0,0,20200705,-0.500783,-0.021146,-0.020828,-0.042827,0.372269,-0.064526,-0.030237,0.020533,-0.029087,-0.048127,-0.040939,-0.007408,0.019336,0.003283,-0.005352,0.007167,0.046009,0.004024,0.008336,0.004438,-0.029383,-0.019759,-0.013523,-0.005138,-0.016134
1,0,20200706,-0.501539,-0.020626,-0.020729,-0.041565,0.377308,-0.065137,-0.030695,0.020893,-0.029217,-0.048340,-0.041060,-0.007169,0.019291,0.003180,-0.005134,0.007255,0.046192,0.004068,0.008484,0.004562,-0.029379,-0.019794,-0.013505,-0.005053,-0.015991
2,0,20200707,-0.502039,-0.020282,-0.020663,-0.040730,0.380641,-0.065541,-0.030998,0.021131,-0.029303,-0.048481,-0.041140,-0.007011,0.019260,0.003112,-0.004990,0.007314,0.046314,0.004096,0.008581,0.004643,-0.029376,-0.019817,-0.013493,-0.004997,-0.015896
3,0,20200708,-0.501037,-0.020974,-0.020795,-0.042404,0.373959,-0.064735,-0.030389,0.020656,-0.029132,-0.048200,-0.040980,-0.007327,0.019318,0.003250,-0.005280,0.007198,0.046070,0.004039,0.008387,0.004479,-0.029382,-0.019771,-0.013516,-0.005109,-0.016086
4,0,20200709,-0.501555,-0.020614,-0.020728,-0.041539,0.377414,-0.065147,-0.030705,0.020899,-0.029222,-0.048345,-0.041063,-0.007165,0.019290,0.003177,-0.005129,0.007252,0.046196,0.004062,0.008486,0.004569,-0.029373,-0.019795,-0.013505,-0.005051,-0.015989
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
343336,10723,20200905,-0.465774,-0.055610,-0.024956,-0.105090,0.165705,-0.048879,-0.016089,0.007375,-0.023988,-0.046700,-0.057562,-0.028377,0.053476,0.018432,-0.031130,-0.010534,0.034007,0.012055,0.154109,-0.045631,-0.103917,-0.208501,0.868223,0.327005,0.103507
343337,10724,20200905,0.500469,0.411354,-0.034175,-0.122323,0.308176,0.722221,-0.538309,-0.015460,0.010847,-0.015498,0.004712,-0.069660,0.012788,-0.003021,-0.003505,-0.017585,0.026633,-0.008742,0.014137,0.020369,-0.016574,-0.009541,-0.001919,0.001508,0.006376
343338,10725,20200905,-0.467286,-0.040927,-0.024133,-0.093091,0.154675,-0.042287,-0.022417,0.010091,-0.017508,-0.035756,-0.030167,-0.019687,0.025565,0.007009,-0.010850,-0.000016,0.021330,0.000719,0.000865,-0.000723,-0.025053,-0.013035,-0.007597,-0.006607,-0.011819
343339,10726,20200905,-0.501116,-0.007233,-0.020317,-0.043395,0.373312,-0.074838,-0.047855,0.018475,-0.022043,-0.048364,-0.034996,-0.020018,0.011682,-0.002065,-0.010802,0.005939,0.040718,0.002515,0.019991,0.008303,-0.029658,-0.023036,-0.010797,-0.000659,-0.014694


In [34]:
grouped_feature = new_item_daily_data.groupby('video_id').mean().drop(columns=['date']).reset_index(level='video_id')
# grouped_feature = grouped_feature.set_index('video_id')
grouped_feature

,video_id,v0,v1,v2,v3,v4,v5,v6,v7,v8,v9,v10,v11,v12,v13,v14,v15,v16,v17,v18,v19,v20,v21,v22,v23,v24
0,0,-0.455352,-0.052904,-0.026822,-0.118887,0.069964,-0.028110,-0.003444,-0.000546,-0.021418,-0.035601,-0.034397,-0.021394,0.020917,0.008985,-0.018723,0.002238,0.036022,0.001595,0.000404,-0.002589,-0.031476,-0.018988,-0.015324,-0.012125,-0.026828
1,1,0.454934,0.262099,0.699175,0.365964,-0.350615,0.245838,0.207768,0.027996,-0.138357,-0.030480,-0.087534,0.388678,-0.309821,-0.262420,0.100628,0.175469,0.579621,-0.176704,-0.030858,-0.252804,0.247271,-0.023172,0.013505,0.020148,0.029147
2,2,0.604627,0.720926,-0.021720,-0.153536,-0.137626,-0.557654,-0.129066,-0.012974,0.035596,-0.021431,-0.006578,-0.041843,0.017866,-0.004676,-0.058936,-0.005695,0.009307,-0.009252,-0.004495,0.004702,-0.029311,-0.020927,-0.007265,-0.009223,-0.002814
3,3,0.498918,0.205528,-0.025525,-0.203114,-0.234439,0.206369,0.264937,0.034318,-0.148255,-0.054614,-0.110811,0.339339,-0.301285,-0.258513,0.075641,0.176351,0.604505,-0.177518,-0.029936,-0.253649,0.224146,-0.036904,0.005818,0.010317,0.014105
4,4,0.904818,-0.615313,0.032611,-0.092350,-0.041057,-0.106685,-0.048657,-0.011760,0.031350,0.015990,-0.088411,-0.062222,0.024511,0.004944,-0.020048,-0.017518,0.003193,-0.003834,-0.032433,-0.079712,-0.139315,-0.098892,-0.403811,0.754125,0.081312
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10723,10723,-0.465774,-0.055610,-0.024956,-0.105090,0.165705,-0.048879,-0.016089,0.007375,-0.023988,-0.046700,-0.057562,-0.028377,0.053476,0.018432,-0.031130,-0.010534,0.034007,0.012055,0.154109,-0.045631,-0.103917,-0.208501,0.868223,0.327005,0.103507
10724,10724,0.500469,0.411354,-0.034175,-0.122323,0.308176,0.722221,-0.538309,-0.015460,0.010847,-0.015498,0.004712,-0.069660,0.012788,-0.003021,-0.003505,-0.017585,0.026633,-0.008742,0.014137,0.020369,-0.016574,-0.009541,-0.001919,0.001508,0.006376
10725,10725,-0.467286,-0.040927,-0.024133,-0.093091,0.154675,-0.042287,-0.022417,0.010091,-0.017508,-0.035756,-0.030167,-0.019687,0.025565,0.007009,-0.010850,-0.000016,0.021330,0.000719,0.000865,-0.000723,-0.025053,-0.013035,-0.007597,-0.006607,-0.011819
10726,10726,-0.501116,-0.007233,-0.020317,-0.043395,0.373312,-0.074838,-0.047855,0.018475,-0.022043,-0.048364,-0.034996,-0.020018,0.011682,-0.002065,-0.010802,0.005939,0.040718,0.002515,0.019991,0.008303,-0.029658,-0.023036,-0.010797,-0.000659,-0.014694


In [36]:
grouped_feature.to_csv("./item_feature_pca_meanday.csv", sep=" ", index=False)